# Side-effect check (control holes only): _ersteSeite_ vs _zweiteSeite_

This notebook is intentionally **small and single-purpose**:  
load only the **control holes** (from the DOE Excel), compute a **short list of robust “side-effect” candidate features**, and test whether flipping the plate introduces a measurable side effect.

---

## What we test (and why)

- We compare **control holes only** (0.5 mm, flagged `ControlSlot=Yes` in the DOE).
- We use **geometry-matched pairing** based on the flip: a 180° rotation in the grid (A1↔G7, …).  
  This pairing matters because the mic is fixed in lab coordinates; after flipping, the *same physical mic-relative* hole positions are mirrored.

We don’t just “fail to reject” differences. We run:
- paired tests (t + Wilcoxon) for sanity
- **bootstrap CIs** for the mean paired difference
- a **TOST equivalence test** with a *predefined* “negligible” margin

If the CI is tight around 0 **and** equivalence holds, it can be called it “negligible” with a straight face.


In [ ]:
# --- imports ---
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal, stats

plt.rcParams["figure.dpi"] = 130

## 1) Configuration

Set the three paths:
- `DOE_XLSX` = the DOE Excel
- `DIR_ERSTE` = folder containing split FLACs for the **first side** (`_ersteSeite_`)
- `DIR_ZWEITE` = folder containing split FLACs for the **second side** (`_zweiteSeite_`)

Expected filename example:
`..._seg001__step001__B2__depth0.500.flac`

If they vary slightly, adjust `FNAME_RE` below.


In [ ]:
# --- paths (edit these) ---
DOE_XLSX = Path(r"./raw_data/Versuchsplan__Bohrungen.xlsx")
DIR_ERSTE = Path(r"./splits_DOE_mapped_padding/05_03_zweitePlatte_ersteSeite_oben5344")
DIR_ZWEITE = Path(r"./splits_DOE_mapped_padding/05_03_zweitePlatte_zweiteSeite_oben3013")

# --- pairing mode ---
# "mirror_col4" = mirror across the middle column (axis A4–G4): A1↔A7, B2↔B6, …
# "rot180"      = full 180° rotation about plate center: A1↔G7, …
# "same"   = same HoleID pairing (only if the labels already compensate the flip)
PAIRING_MODE = "same"

# --- equivalence margin (negligible effect threshold) ---
# For normalized features (fractions, ratios), 0.03–0.05 is often “small”.
# Adjust if one has a real engineering tolerance.
EQUIV_DELTA_DEFAULT = 0.05

# --- analysis settings ---
BOOTSTRAP_N = 20_000
CI_LEVEL = 0.95
RANDOM_SEED = 7

## 2) Load DOE and extract control holes

The DOE Excel includes:
- `Step` (run order)
- `HoleID` (A1..G7)
- `ControlSlot` (Yes/NaN)

We filter to `ControlSlot == "Yes"` and use those steps to pick the right FLACs.


In [ ]:
df_doe = pd.read_excel(DOE_XLSX, sheet_name="DOE_run_order")

required_cols = {"Step", "HoleID", "Depth_mm", "ControlSlot"}
missing = required_cols - set(df_doe.columns)
if missing:
    raise ValueError(f"DOE sheet is missing columns: {missing}. Found: {list(df_doe.columns)}")

df_controls = df_doe[df_doe["ControlSlot"].astype(str).str.lower().eq("yes")].copy()
df_controls = df_controls.sort_values("Step")

print(f"DOE rows: {len(df_doe)}")
print(f"Control holes: {len(df_controls)}")
display(df_controls[["Step", "HoleID", "Depth_mm", "Note", "X_mm", "Y_mm"]])

## 3) Discover FLAC files and parse metadata

We crawl the two directories and parse:
- step number
- HoleID (A1..G7)
- depth

If parsing yields 0 files, it’s either a wrong folder or the regex is too strict.


In [ ]:
# Regex for names like:
# 05_03_..._seg001__step001__B2__depth0.500.flac
FNAME_RE = re.compile(
    r".*?_seg(?P<seg>\d+).*?step(?P<step>\d+).*?__(?P<hole>[A-G][1-7])__depth(?P<depth>\d+\.\d+)\.flac$",
    re.IGNORECASE,
)


def find_flacs(root: Path) -> pd.DataFrame:
    paths = sorted(root.rglob("*.flac"))
    rows = []
    for p in paths:
        m = FNAME_RE.match(p.name)
        if not m:
            continue
        rows.append(
            {
                "path": p,
                "seg": int(m.group("seg")),
                "step": int(m.group("step")),
                "hole": m.group("hole").upper(),
                "depth": float(m.group("depth")),
            }
        )
    return pd.DataFrame(rows)


df_e = find_flacs(DIR_ERSTE)
df_z = find_flacs(DIR_ZWEITE)

print("ersteSeite files parsed:", len(df_e))
print("zweiteSeite files parsed:", len(df_z))

display(df_e.head(5))
display(df_z.head(5))

## 4) Build geometry-matched pairs (control holes only)

### Flip model: 180° rotation on the 7×7 grid
- letters: A↔G, B↔F, C↔E, D↔D
- numbers: 1↔7, 2↔6, 3↔5, 4↔4

This is the pairing that is wanted if the mic stays fixed and the plate is flipped.

We also support `PAIRING_MODE="same"` as a fallback.


In [ ]:
LETTERS = "ABCDEFG"


def rot180_hole(h: str) -> str:
    """180° rotation about plate center (A1↔G7)."""
    h = h.strip().upper()
    letter = h[0]
    number = int(h[1])
    li = LETTERS.index(letter)
    ni = number - 1
    li2 = 6 - li
    ni2 = 6 - ni
    return f"{LETTERS[li2]}{ni2 + 1}"


def mirror_col4_hole(h: str) -> str:
    """Mirror across middle column (axis A4–G4): A1↔A7, A2↔A6, A3↔A5, A4↔A4."""
    h = h.strip().upper()
    letter = h[0]
    number = int(h[1])
    return f"{letter}{8 - number}"


def map_hole(h: str, mode: str) -> str:
    m = (mode or "").lower().strip()
    if m == "mirror_col4":
        return mirror_col4_hole(h)
    if m == "rot180":
        return rot180_hole(h)
    return h.strip().upper()


# show mapping for the control holes
pairs_preview = []
for h in df_controls["HoleID"].astype(str):
    pairs_preview.append((h.upper(), map_hole(h, PAIRING_MODE)))
display(pd.DataFrame(pairs_preview, columns=["HoleID", f"mapped({PAIRING_MODE})"]))

In [ ]:
# Filter to control steps (more reliable than depth==0.5 alone)
control_steps = set(df_controls["Step"].astype(int).tolist())
control_holes = set(df_controls["HoleID"].astype(str).str.upper().tolist())

df_e_c = df_e[df_e["step"].isin(control_steps)].copy()
df_z_c = df_z[df_z["step"].isin(control_steps)].copy()

# Optional: keep only depth==0.5 as a sanity check
df_e_c = df_e_c[np.isclose(df_e_c["depth"], 0.5, atol=1e-6)]
df_z_c = df_z_c[np.isclose(df_z_c["depth"], 0.5, atol=1e-6)]

print("control files (erste):", len(df_e_c))
print("control files (zweite):", len(df_z_c))

# Index for matching; if duplicates exist, we pick the lowest seg
idx_e = df_e_c.sort_values("seg").groupby("hole").first()
idx_z = df_z_c.sort_values("seg").groupby("hole").first()

pairs = []
missing = []

for h in sorted(control_holes):
    h_e = h
    h_z = map_hole(h, PAIRING_MODE)
    if h_e in idx_e.index and h_z in idx_z.index:
        pairs.append(
            {
                "hole_e": h_e,
                "hole_z": h_z,
                "path_e": idx_e.loc[h_e, "path"],
                "path_z": idx_z.loc[h_z, "path"],
            }
        )
    else:
        missing.append((h_e, h_z))

df_pairs = pd.DataFrame(pairs)
print("matched pairs:", len(df_pairs))
if missing:
    print("missing pairs (hole_e -> hole_z):", missing)

display(df_pairs)

## 5) Feature extraction (small, robust set)

We compute mostly **shape / spectrum** features that are less sensitive to mic distance:

- **Band power fractions** (power in band / total power)
- **Spectral centroid ratio** (centroid / max_freq)
- **Rolloff ratio** (85% rolloff / max_freq)
- **Spectral flatness**
- **Peak frequency ratio** (peak freq / max_freq)
- **Peak-to-total power ratio**

We also compute `rms` as a “setup sensitivity” indicator.


In [ ]:
def read_audio(path: Path) -> tuple[np.ndarray, int]:
    y, sr = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float64)
    y = y - np.mean(y)
    return y, int(sr)


def welch_psd(y: np.ndarray, sr: int, nperseg: int = 8192) -> tuple[np.ndarray, np.ndarray]:
    nperseg = min(nperseg, len(y))
    if nperseg < 1024:
        nperseg = max(256, nperseg)
    f, pxx = signal.welch(
        y, fs=sr, nperseg=nperseg, noverlap=nperseg // 2, window="hann", scaling="density"
    )
    return f, pxx


def spectral_features_from_psd(f: np.ndarray, pxx: np.ndarray, fmax: float = 20000.0) -> dict:
    fmax_eff = min(float(fmax), float(f.max()))
    mask = (f >= 0) & (f <= fmax_eff)
    f2 = f[mask]
    p2 = pxx[mask] + 1e-18

    total = float(np.trapz(p2, f2))
    if total <= 0:
        total = float(np.sum(p2))

    bands = [(0, 2000), (2000, 6000), (6000, 12000), (12000, 20000)]
    out = {}
    for lo, hi in bands:
        hi_eff = min(hi, fmax_eff)
        m = (f2 >= lo) & (f2 < hi_eff)
        bp = float(np.trapz(p2[m], f2[m])) if m.any() else 0.0
        out[f"bp_frac_{int(lo)}_{int(hi)}"] = bp / total

    centroid = float(np.sum(f2 * p2) / np.sum(p2))
    out["centroid_ratio"] = centroid / fmax_eff

    cumsum = np.cumsum(p2)
    roll_idx = int(np.searchsorted(cumsum, 0.85 * cumsum[-1]))
    roll_idx = min(roll_idx, len(f2) - 1)
    out["rolloff85_ratio"] = float(f2[roll_idx] / fmax_eff)

    out["flatness"] = float(np.exp(np.mean(np.log(p2))) / np.mean(p2))

    peak_idx = int(np.argmax(p2))
    peak_f = float(f2[peak_idx])
    out["peak_freq_ratio"] = peak_f / fmax_eff
    out["peak_power_ratio"] = float(p2[peak_idx] / np.sum(p2))

    return out


def extract_features(path: Path, fmax: float = 20000.0) -> dict:
    y, sr = read_audio(path)
    f, pxx = welch_psd(y, sr)
    feats = spectral_features_from_psd(f, pxx, fmax=fmax)
    feats["rms"] = float(np.sqrt(np.mean(y**2)))
    feats["sr"] = int(sr)
    feats["n_samples"] = int(len(y))
    return feats

In [ ]:
# Extract features for each paired file
rows = []
for r in df_pairs.itertuples(index=False):
    fe = extract_features(Path(r.path_e))
    fz = extract_features(Path(r.path_z))

    rows.append({"side": "erste", "hole": r.hole_e, "path": str(r.path_e), **fe})
    rows.append({"side": "zweite", "hole": r.hole_z, "path": str(r.path_z), **fz})

df_feat = pd.DataFrame(rows)

display(df_feat[["side", "hole", "sr", "n_samples", "rms"]].sort_values(["side", "hole"]))

## 6) Stats: paired tests + bootstrap CI + equivalence (TOST) + FDR

For each feature:
- paired difference = zweite − erste
- paired t-test + Wilcoxon
- bootstrap CI for mean difference
- equivalence test (TOST) within ±delta

We also compute Benjamini–Hochberg adjusted p-values (FDR) as a sanity check.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)


def bootstrap_ci_mean(d: np.ndarray, alpha: float = 0.05, n: int = 20000) -> tuple[float, float]:
    d = np.asarray(d, float)
    idx = rng.integers(0, len(d), size=(n, len(d)))
    means = d[idx].mean(axis=1)
    lo = float(np.quantile(means, alpha / 2))
    hi = float(np.quantile(means, 1 - alpha / 2))
    return lo, hi


def tost_paired(d: np.ndarray, delta: float, alpha: float = 0.05) -> dict:
    d = np.asarray(d, float)
    n = len(d)
    md = float(d.mean())
    sd = float(d.std(ddof=1))
    se = sd / np.sqrt(n)
    df = n - 1

    # H01: mean <= -delta  vs H11: mean > -delta
    t1 = (md + delta) / se
    p1 = float(1 - stats.t.cdf(t1, df=df))

    # H02: mean >= +delta  vs H12: mean < +delta
    t2 = (md - delta) / se
    p2 = float(stats.t.cdf(t2, df=df))

    equiv = (p1 < alpha) and (p2 < alpha)
    return {"tost_p1": p1, "tost_p2": p2, "equivalent": bool(equiv)}


def bh_fdr(p: np.ndarray) -> np.ndarray:
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan, dtype=float)
    m = np.isfinite(p)
    pv = p[m]
    if pv.size == 0:
        return out
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adj = ranked * n / (np.arange(1, n + 1))
    # enforce monotonicity
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out_idx = np.where(m)[0][order]
    out[out_idx] = adj
    return out


feat_cols = [c for c in df_feat.columns if c not in {"side", "hole", "path", "sr", "n_samples"}]


def aligned_arrays(feature: str) -> tuple[np.ndarray, np.ndarray]:
    xs, ys = [], []
    for r in df_pairs.itertuples(index=False):
        x = df_feat[(df_feat["side"] == "erste") & (df_feat["hole"] == r.hole_e)][feature].values
        y = df_feat[(df_feat["side"] == "zweite") & (df_feat["hole"] == r.hole_z)][feature].values
        if len(x) == 1 and len(y) == 1:
            xs.append(x[0])
            ys.append(y[0])
    return np.asarray(xs, float), np.asarray(ys, float)


results = []
alpha = 1 - CI_LEVEL

for fcol in feat_cols:
    x, y = aligned_arrays(fcol)
    d = y - x
    n = len(d)

    t_p = float(stats.ttest_rel(y, x).pvalue) if n >= 2 else np.nan
    try:
        w_p = float(stats.wilcoxon(d).pvalue) if n >= 2 else np.nan
    except Exception:
        w_p = np.nan

    ci_lo, ci_hi = bootstrap_ci_mean(d, alpha=alpha, n=BOOTSTRAP_N) if n >= 2 else (np.nan, np.nan)

    delta = float(EQUIV_DELTA_DEFAULT)
    tost = (
        tost_paired(d, delta=delta, alpha=0.05)
        if n >= 3
        else {"tost_p1": np.nan, "tost_p2": np.nan, "equivalent": False}
    )

    dz = float(d.mean() / d.std(ddof=1)) if n >= 3 and d.std(ddof=1) > 0 else np.nan

    results.append(
        {
            "feature": fcol,
            "n_pairs": n,
            "mean_diff": float(d.mean()) if n else np.nan,
            f"ci{int(CI_LEVEL * 100)}_lo": ci_lo,
            f"ci{int(CI_LEVEL * 100)}_hi": ci_hi,
            "p_ttest_paired": t_p,
            "p_wilcoxon": w_p,
            "cohens_dz": dz,
            "equiv_delta": delta,
            "tost_p1": tost["tost_p1"],
            "tost_p2": tost["tost_p2"],
            "equivalent": tost["equivalent"],
        }
    )

df_res = pd.DataFrame(results)
df_res["p_ttest_fdr_bh"] = bh_fdr(df_res["p_ttest_paired"].values)

df_res = df_res.sort_values("feature").reset_index(drop=True)
display(df_res)

## 7) Visual sanity checks

Two plots per feature:
1) **erste vs zweite** scatter with identity line  
2) paired differences histogram (zweite − erste)

If “side effect is negligible”, one will see points near the diagonal and differences clustered near 0.


In [ ]:
def plot_feature(feature: str):
    x, y = aligned_arrays(feature)
    d = y - x

    plt.figure(figsize=(10, 4))

    ax1 = plt.subplot(1, 2, 1)
    ax1.scatter(x, y)
    lo = float(min(x.min(), y.min()))
    hi = float(max(x.max(), y.max()))
    ax1.plot([lo, hi], [lo, hi], linestyle="--")
    ax1.set_xlabel("erste")
    ax1.set_ylabel("zweite")
    ax1.set_title(f"{feature}: erste vs zweite")

    ax2 = plt.subplot(1, 2, 2)
    ax2.hist(d, bins=max(5, len(d)))
    ax2.axvline(0, linestyle="--")
    ax2.set_title(f"{feature}: diff (zweite − erste)")
    ax2.set_xlabel("difference")

    plt.tight_layout()
    plt.show()


key_feats = [
    "bp_frac_0_2000",
    "bp_frac_2000_6000",
    "bp_frac_6000_12000",
    "centroid_ratio",
    "rolloff85_ratio",
    "flatness",
    "peak_freq_ratio",
    "peak_power_ratio",
    "rms",
]

for k in key_feats:
    if k in feat_cols:
        plot_feature(k)

## 8) Interpretation helper (auto-summary)

This prints one line per feature:
- mean diff + CI
- equivalence pass/fail
- p-values (paired t and Wilcoxon)
- FDR-adjusted p-value (paired t)

The “ironclad” part is: **tight CI around 0 + equivalence accepted**.


In [ ]:
def summarize_results(df_res: pd.DataFrame) -> str:
    ci_lo_name = f"ci{int(CI_LEVEL * 100)}_lo"
    ci_hi_name = f"ci{int(CI_LEVEL * 100)}_hi"

    lines = []
    for r in df_res.itertuples(index=False):
        ci_lo = getattr(r, ci_lo_name)
        ci_hi = getattr(r, ci_hi_name)
        lines.append(
            f"- {r.feature}: mean diff={r.mean_diff:+.4f} "
            f"[{ci_lo:+.4f}, {ci_hi:+.4f}], "
            f"equiv(±{r.equiv_delta})={r.equivalent}, "
            f"p_t={r.p_ttest_paired:.4g}, p_w={r.p_wilcoxon:.4g}, "
            f"p_t(FDR)={r.p_ttest_fdr_bh:.4g}"
        )
    return "\n".join(lines)


print(summarize_results(df_res))